In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS music_diary
COMMENT 'Database container for ListenBrainz and Last.fm ingestion pipeline';

In [0]:
%sql
-- Raw dump of paginated ListenBrainz user stream history
CREATE TABLE IF NOT EXISTS music_diary.bronze_listens_raw (
    raw_payload STRING,         -- The raw JSON payload string
    ingested_at TIMESTAMP,      -- System clock timestamp of ingestion run
    source_file_name STRING     -- Traceability back to pipeline run logs
)
USING DELTA
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true, delta.autoOptimize.autoCompact = true);

-- Raw dump of MusicBrainz API responses (Artist search, Release details, Recording data)
CREATE TABLE IF NOT EXISTS music_diary.bronze_musicbrainz_raw (
    mbid STRING,                -- The MusicBrainz ID acting as the lookup key
    query_term STRING,          -- The original string searched (useful if fallback text matching was used)
    entity_type STRING,         -- 'artist', 'release', 'release-group', or 'recording'
    raw_payload STRING,         -- The complete raw XML/JSON response from the API
    ingested_at TIMESTAMP       -- Clock timestamp when the API call was recorded
)
USING DELTA
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

-- Raw dump of Last.fm entity enrichment responses
CREATE TABLE IF NOT EXISTS music_diary.bronze_lastfm_raw (
    entity_id STRING,           -- Can be artist_id, release_id, or track_id
    entity_type STRING,         -- 'artist', 'album', or 'track'
    raw_payload STRING,         -- Raw JSON metadata return from Last.fm
    ingested_at TIMESTAMP
)
USING DELTA;


In [0]:
%sql
-- Raw dump of Wikidata genre hierarchy API responses
CREATE TABLE IF NOT EXISTS music_diary.bronze_wikidata_genres (
    wikidata_id STRING,              -- The Wikidata entity ID (e.g., 'Q11399')
    musicbrainz_genre STRING,        -- The original genre string from MusicBrainz
    raw_payload STRING,              -- Complete Wikidata JSON response with all properties
    ingested_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS music_diary.silver_artists (
    artist_mbid STRING,                -- Unique natural identifier from MusicBrainz
    artist_name STRING,                -- Official clean name string
    normalized_artist_name STRING,     -- Handled by your vectorized text normalization UDF
    sort_name STRING,                  -- MusicBrainz sort layout (e.g., "Stone, Meaningful")
    artist_type STRING,                -- Person, Group, Orchestra, etc.
    country STRING,                    -- 2-character ISO country code
    aliases ARRAY<STRING>,             -- Combines your old artist_aliases table into a fast array
    genres ARRAY<STRING>,              -- Tag strings extracted directly from Last.fm / MusicBrainz
    bio_summary STRING,                -- Text summary extracted from Last.fm profile lookups
    last_updated_at TIMESTAMP          -- Tracking marker for metadata refresh loops
)
USING DELTA
TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true);

CREATE TABLE IF NOT EXISTS music_diary.silver_releases (
    release_mbid STRING,               -- MusicBrainz Release ID
    release_group_mbid STRING,         -- MusicBrainz Release Group ID (useful for grouping deluxe editions)
    release_name STRING,               -- Official album name
    normalized_release_name STRING,    -- UDF-normalized name for matching runs
    primary_artist_mbid STRING,        -- Relationship pointer back to silver_artists
    primary_artist_name STRING,        -- Included directly to avoid expensive SQL JOIN operations
    release_type STRING,               -- Album, Single, EP, Compilation, Live
    release_date DATE,                 -- Parsed into clean date formatting
    country STRING,                    -- Release distribution region
    genres ARRAY<STRING>,              -- List of album genres parsed from Last.fm toptags
    last_updated_at TIMESTAMP
)
USING DELTA;

CREATE TABLE IF NOT EXISTS music_diary.silver_tracks (
    recording_mbid STRING,             -- MusicBrainz Recording ID
    release_mbid STRING,               -- Reference back to parent album
    track_name STRING,                 -- Clean track name string
    normalized_track_name STRING,      -- Clean text representation for validation passes
    track_number INT,                  -- Numerical track order index on the album
    duration_ms LONG,                  -- Track duration parsed out from MusicBrainz metadata
    last_updated_at TIMESTAMP
)
USING DELTA;

CREATE OR REPLACE TABLE music_diary.silver_listens (
    listen_timestamp TIMESTAMP,                             -- The exact point in time the listen occurred
    listen_date DATE GENERATED ALWAYS AS (CAST(listen_timestamp AS DATE)),  -- Auto-computed partition column
    
    -- Normalized matched identifiers (Keys connecting to dimension tables)
    artist_mbid STRING,                -- Nullable if fuzzy validation fails
    release_mbid STRING,               -- Nullable if fuzzy validation fails
    recording_mbid STRING,             -- Nullable if fuzzy validation fails
    
    -- Raw text capture strings straight from the ingestion payload
    raw_track_name STRING,
    raw_artist_name STRING,
    raw_release_name STRING,
    
    -- System lineage tracking
    ingested_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (listen_date);

In [0]:
%sql
-- Normalized genre hierarchy with parent relationships
CREATE TABLE IF NOT EXISTS music_diary.silver_genre_hierarchy (
    genre_name STRING,               -- Standardized genre name (from Wikidata label)
    wikidata_id STRING,              -- Wikidata entity ID
    parent_genre_name STRING,        -- Parent genre (e.g., 'hip hop' for 'east coast hip hop')
    parent_wikidata_id STRING,       -- Parent's Wikidata ID
    genre_level INT,                 -- Depth in hierarchy (0=root, 1=subgenre, 2=sub-subgenre, etc.)
    is_root_genre BOOLEAN,           -- TRUE for top-level genres (no parent)
    last_updated_at TIMESTAMP
)
USING DELTA;

-- Mapping table: connects raw MusicBrainz/Last.fm genres to standardized hierarchy
CREATE TABLE IF NOT EXISTS music_diary.silver_genre_mapping (
    source_genre STRING,             -- Raw genre string from MusicBrainz/Last.fm
    source_system STRING,            -- 'musicbrainz' or 'lastfm'
    mapped_genre_name STRING,        -- Points to silver_genre_hierarchy.genre_name
    wikidata_id STRING,              -- Wikidata ID for joining to hierarchy
    mapping_confidence STRING,       -- 'exact', 'fuzzy', or 'manual' (if you curate overrides)
    last_updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS music_diary.silver_popularity_snapshots (
    entity_mbid STRING,                -- Can map to an artist_mbid, release_mbid, or recording_mbid
    entity_type STRING,                -- 'artist', 'release', or 'track'
    listeners LONG,                    -- Absolute audience depth metric from Last.fm
    playcount LONG,                    -- Total historical platform plays from Last.fm
    snapshot_date DATE                 -- The day this record was logged
)
USING DELTA
PARTITIONED BY (snapshot_date);